# 11 — RAG Baseline: Few-Shot Retrieval vs Fine-Tuning

**Goal**: Evaluate a RAG approach (retrieve similar training examples → few-shot prompt → base Llama-3) as a baseline to compare against our fine-tuned model.

**Method**: For each test question, we:
1. Embed the question + schema with `all-MiniLM-L6-v2`
2. Retrieve top-3 similar training examples from ChromaDB
3. Build a few-shot prompt with those examples as demonstrations
4. Send to **base** Llama-3-8B-Instruct (no fine-tuning, no chat prefix)
5. Evaluate with our standard execution accuracy pipeline

**Runtime**: Google Colab T4 GPU (~1-2 hours for 500 examples)

## 0. Setup

In [1]:
# Install dependencies
!pip install -q transformers peft bitsandbytes accelerate torch
!pip install -q "datasets<3.0"
!pip install -q chromadb sentence-transformers langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 19.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [2]:
# Clone the repo (or mount Google Drive if preferred)
!git clone https://github.com/oLittle-five/text-to-sql-llama.git
%cd text-to-sql-llama

Cloning into 'text-to-sql-llama'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 131 (delta 55), reused 116 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 5.04 MiB | 4.42 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/text-to-sql-llama


In [3]:
import json
import time
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

## 1. Build ChromaDB Index on Colab

We regenerate the training JSONL and build the vector index directly on Colab.
This avoids uploading the ~200MB ChromaDB folder.

In [4]:
# Generate training JSONL from WikiSQL
from src.data.prepare_dataset import prepare_and_save
prepare_and_save("data/processed")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository for Salesforce/wikisql contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/Salesforce/wikisql.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating test split:   0%|          | 0/15878 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8421 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/56355 [00:00<?, ? examples/s]

Saved 56355 examples to data/processed/train.jsonl
Saved 8421 examples to data/processed/validation.jsonl
Saved 15878 examples to data/processed/test.jsonl


In [5]:
# Build the ChromaDB index (~5-10 min on Colab)
from src.rag.build_index import build_index
build_index("data/processed/train.jsonl", "data/chroma_db")

Loading training examples from data/processed/train.jsonl ...
  Loaded 56,355 examples
Initializing embedding model: all-MiniLM-L6-v2 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building documents and embedding ...
  Added 512 / 56,355 examples
  Added 10,752 / 56,355 examples
  Added 20,992 / 56,355 examples
  Added 31,232 / 56,355 examples
  Added 41,472 / 56,355 examples
  Added 51,712 / 56,355 examples
  Added 56,355 / 56,355 examples

Done! ChromaDB index saved to data/chroma_db/
  Collection: wikisql_train
  Documents:  56,355


In [6]:
# Initialize the RAG pipeline
from src.rag.rag_pipeline import RAGPipeline
rag = RAGPipeline(db_dir="data/chroma_db", top_k=3)

RAG pipeline loaded: 56,355 examples in index


## 2. Quick Retrieval Sanity Check

Verify that retrieved examples look relevant before running full evaluation.

In [7]:
# Load test set
ds = load_dataset("Salesforce/wikisql", trust_remote_code=True)
test_examples = list(ds["test"].select(range(500)))
print(f"Test examples: {len(test_examples)}")

Test examples: 500


In [8]:
# Preview: retrieve examples for the first test question
ex = test_examples[0]
print(f"Test question: {ex['question']}")
print(f"Test columns:  {ex['table']['header']}")
print()

retrieved = rag.retrieve(ex["question"], ex["table"]["header"], ex["table"]["types"])
for i, r in enumerate(retrieved, 1):
    print(f"--- Example {i} (distance: {r['distance']:.4f}) ---")
    print(r["text"][:200])
    print()

Test question: What is terrence ross' nationality
Test columns:  ['Player', 'No.', 'Nationality', 'Position', 'Years in Toronto', 'School/Club Team']

--- Example 1 (distance: 0.1188) ---
### Input:
Columns: Player (text), No. (text), Nationality (text), Position (text), Years in Toronto (text), School/Club Team (text)

Question: What nationality is the player Muggsy Bogues?

### SQL:


--- Example 2 (distance: 0.1254) ---
### Input:
Columns: Player (text), No. (real), Nationality (text), Position (text), Years in Toronto (text), School/Club Team (text)

Question: what nationality is the player who played from 1997-98



--- Example 3 (distance: 0.1375) ---
### Input:
Columns: Player (text), Nationality (text), Position (text), Years in Toronto (text), School/Club Team (text)

Question: What is the nationality of Martin Lewis?

### SQL:
SELECT `Nationali



In [9]:
# Preview the full few-shot prompt for the first test example
prompt = rag.build_few_shot_prompt(
    ex["question"], ex["table"]["header"], ex["table"]["types"]
)
print(prompt)
print(f"\nPrompt length: {len(prompt)} chars")

Below are examples of converting natural language questions to SQL queries.

[Example 1]
### Input:
Columns: Player (text), No. (text), Nationality (text), Position (text), Years in Toronto (text), School/Club Team (text)

Question: What nationality is the player Muggsy Bogues?

### SQL:
SELECT `Nationality` FROM table WHERE `Player` = 'Muggsy Bogues'

[Example 2]
### Input:
Columns: Player (text), No. (real), Nationality (text), Position (text), Years in Toronto (text), School/Club Team (text)

Question: what nationality is the player who played from 1997-98

### SQL:
SELECT `Nationality` FROM table WHERE `Years in Toronto` = '1997-98'

[Example 3]
### Input:
Columns: Player (text), Nationality (text), Position (text), Years in Toronto (text), School/Club Team (text)

Question: What is the nationality of Martin Lewis?

### SQL:
SELECT `Nationality` FROM table WHERE `Player` = 'martin lewis'

Now generate SQL for this question:

### Input:
Columns: Player (text), No. (text), Nationalit

## 3. Load Base Model

In [10]:
from huggingface_hub import login
login()

In [12]:
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
print("Model loaded.")

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded.


In [13]:
# Generation config for base model (NO repetition_penalty — same as notebook 06)
GENERATION_CONFIG = dict(
    max_new_tokens=128,
    do_sample=False,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=[
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    ],
)

## 4. SQL Generation with RAG

In [14]:
import re

def extract_sql(response: str, columns: list[str]) -> str:
    """
    Extract the SQL query from the model's response.
    """
    text = response.strip()

    # Strategy 1: Find a SELECT statement directly
    select_match = re.search(r'(SELECT\s.+?)(?:\n\n|```|;|$)', text, re.IGNORECASE | re.DOTALL)
    if select_match:
        sql = select_match.group(1).strip().rstrip(';')
        # Collapse multi-line SQL into single line
        sql = " ".join(line.strip() for line in sql.split("\n") if line.strip())
        return sql

    # Strategy 2: Extract from code fences (skip empty blocks)
    code_blocks = re.findall(r'```(?:sql)?\s*\n?(.*?)```', text, re.DOTALL | re.IGNORECASE)
    for block in code_blocks:
        block = block.strip()
        if block and 'SELECT' in block.upper():
            sql = " ".join(line.strip() for line in block.split("\n")
                          if line.strip() and not line.strip().startswith("--"))
            return sql

    # Strategy 3: fallback — return first non-empty line
    for line in text.split("\n"):
        line = line.strip()
        if line and not line.startswith("```") and not line.startswith("--"):
            return line

    return text.split("\n")[0].strip() if text else ""


def generate_sql_rag(model, tokenizer, rag_pipeline, question, columns, types, gen_config):
    """Generate SQL using RAG few-shot prompting."""
    prompt = rag_pipeline.build_few_shot_prompt(question, columns, types)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_config)

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return extract_sql(response, columns)

In [15]:
# Quick test: generate SQL for the first example
ex = test_examples[0]
pred_sql = generate_sql_rag(
    model, tokenizer, rag,
    ex["question"], ex["table"]["header"], ex["table"]["types"],
    GENERATION_CONFIG,
)
print(f"Question:  {ex['question']}")
print(f"Predicted: {pred_sql}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question:  What is terrence ross' nationality
Predicted: SELECT `Nationality` FROM table WHERE `Player` = 'terrence ross'


In [16]:
ex = test_examples[0]
prompt = rag.build_few_shot_prompt(ex["question"], ex["table"]["header"], ex["table"]["types"])
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, **GENERATION_CONFIG)

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
print(f"New tokens count: {len(new_tokens)}")
print(f"Raw token IDs: {new_tokens.tolist()}")
raw_response = tokenizer.decode(new_tokens, skip_special_tokens=False)
print(f"Raw response (with special tokens): repr = {repr(raw_response)}")

New tokens count: 51
Raw token IDs: [14196, 4077, 74694, 3628, 198, 4963, 1595, 31912, 488, 63, 720, 31193, 2007, 720, 27611, 1595, 4576, 63, 284, 364, 466, 16271, 938, 784, 1270, 14196, 19884, 14196, 19884, 5618, 1095, 757, 1440, 422, 499, 617, 904, 4726, 7540, 0, 358, 4265, 387, 6380, 311, 1520, 449, 810, 49822, 13, 128009]
Raw response (with special tokens): repr = "```\n```sql\nSELECT `Nationality` \nFROM table \nWHERE `Player` = 'terrence ross'\n```\n\n```\n\nPlease let me know if you have any further requests! I'd be happy to help with more conversions.<|eot_id|>"


## 5. Run Full Evaluation (500 examples)

In [17]:
N_EXAMPLES = 500

predictions = []
start_time = time.time()

for i, ex in enumerate(tqdm(test_examples[:N_EXAMPLES], desc="RAG inference")):
    pred_sql = generate_sql_rag(
        model, tokenizer, rag,
        ex["question"], ex["table"]["header"], ex["table"]["types"],
        GENERATION_CONFIG,
    )
    predictions.append(pred_sql)

    # Progress check every 50 examples
    if (i + 1) % 50 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (N_EXAMPLES - i - 1) / rate
        print(f"  [{i+1}/{N_EXAMPLES}] {elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining")

total_time = time.time() - start_time
print(f"\nDone! {N_EXAMPLES} examples in {total_time:.0f}s ({total_time/N_EXAMPLES:.1f}s/example)")

RAG inference:  10%|█         | 50/500 [06:47<55:48,  7.44s/it]

  [50/500] 408s elapsed, ~3668s remaining


RAG inference:  20%|██        | 100/500 [13:03<38:33,  5.78s/it]

  [100/500] 784s elapsed, ~3134s remaining


RAG inference:  30%|███       | 150/500 [19:38<33:21,  5.72s/it]

...

  [400/500] 3228s elapsed, ~807s remaining


RAG inference:  90%|█████████ | 450/500 [1:00:14<06:34,  7.90s/it]

  [450/500] 3615s elapsed, ~402s remaining


RAG inference: 100%|██████████| 500/500 [1:07:01<00:00,  8.04s/it]

  [500/500] 4021s elapsed, ~0s remaining

Done! 500 examples in 4021s (8.0s/example)


## 6. Evaluate with Execution Accuracy

In [18]:
from src.data.prepare_dataset import build_sql_string
from src.data.sql_executor import execute_sql

def normalize_result(result):
    """Normalize SQL result for comparison."""
    if result is None:
        return None
    return sorted([tuple(str(v).lower().strip() for v in row) for row in result])


def evaluate(predictions, examples):
    """Compute execution accuracy."""
    total = len(predictions)
    correct = syntax_errors = wrong_result = 0

    for pred, ex in zip(predictions, examples):
        table = ex["table"]
        gold_sql = build_sql_string(ex["sql"], table["header"], table["types"])

        pred_results, pred_error = execute_sql(table, pred)
        gold_results, gold_error = execute_sql(table, gold_sql)

        if pred_error:
            syntax_errors += 1
        elif normalize_result(pred_results) == normalize_result(gold_results):
            correct += 1
        else:
            wrong_result += 1

    return {
        "execution_accuracy": correct / total,
        "syntax_error_rate": syntax_errors / total,
        "wrong_result_rate": wrong_result / total,
        "correct": correct,
        "syntax_errors": syntax_errors,
        "wrong_results": wrong_result,
        "total": total,
    }


results = evaluate(predictions, test_examples[:N_EXAMPLES])

print("=" * 50)
print("RAG BASELINE RESULTS")
print("=" * 50)
print(f"Execution accuracy: {results['execution_accuracy']:.1%} ({results['correct']}/{results['total']})")
print(f"Syntax error rate:  {results['syntax_error_rate']:.1%} ({results['syntax_errors']}/{results['total']})")
print(f"Wrong result rate:  {results['wrong_result_rate']:.1%} ({results['wrong_results']}/{results['total']})")

RAG BASELINE RESULTS
Execution accuracy: 44.0% (220/500)
Syntax error rate:  19.6% (98/500)
Wrong result rate:  36.4% (182/500)


## 7. Compare All Approaches

In [19]:
print("\n" + "=" * 70)
print("COMPARISON: ALL APPROACHES")
print("=" * 70)
print(f"{'Approach':<40} {'Exec Acc':>10} {'Syntax Err':>12}")
print("-" * 70)
print(f"{'Base model (zero-shot)':<40} {'37.2%':>10} {'21.8%':>12}")
print(f"{'RAG baseline (3-shot retrieval)':<40} {results['execution_accuracy']:>10.1%} {results['syntax_error_rate']:>12.1%}")
print(f"{'Fine-tuned v1 (chat template fix)':<40} {'51.8%':>10} {'5.4%':>12}")
print(f"{'Fine-tuned v1 + COLLATE NOCASE':<40} {'68.2%':>10} {'5.4%':>12}")
print("=" * 70)


COMPARISON: ALL APPROACHES
Approach                                   Exec Acc   Syntax Err
----------------------------------------------------------------------
Base model (zero-shot)                        37.2%        21.8%
RAG baseline (3-shot retrieval)               44.0%        19.6%
Fine-tuned v1 (chat template fix)             51.8%         5.4%
Fine-tuned v1 + COLLATE NOCASE                68.2%         5.4%


## 8. Save Results

In [20]:
import os
os.makedirs("results", exist_ok=True)

rag_save = {
    "experiment": "RAG baseline (3-shot retrieval)",
    "model": MODEL_ID,
    "adapter": None,
    "method": "RAG few-shot",
    "retrieval_model": "all-MiniLM-L6-v2",
    "top_k": 3,
    "prompt_format": "few-shot with retrieved examples",
    "generation_config": {k: str(v) for k, v in GENERATION_CONFIG.items()},
    "n_examples": N_EXAMPLES,
    "execution_accuracy": results["execution_accuracy"],
    "syntax_error_rate": results["syntax_error_rate"],
    "wrong_result_rate": results["wrong_result_rate"],
    "correct": results["correct"],
    "syntax_errors": results["syntax_errors"],
    "wrong_results": results["wrong_results"],
    "total_time_seconds": round(total_time, 1),
    "predictions": predictions,
}

with open("results/rag_results.json", "w") as f:
    json.dump(rag_save, f, indent=2)

print("Saved to results/rag_results.json")

Saved to results/rag_results.json


In [21]:
# Download results to your local machine
from google.colab import files
files.download("results/rag_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Sample Predictions

Inspect a few predictions to see how the RAG approach works.

In [22]:
# Show 10 sample predictions
for i in range(10):
    ex = test_examples[i]
    gold = build_sql_string(ex["sql"], ex["table"]["header"], ex["table"]["types"])
    pred = predictions[i]

    # Check correctness
    pred_res, pred_err = execute_sql(ex["table"], pred)
    gold_res, _ = execute_sql(ex["table"], gold)
    status = "SYNTAX_ERR" if pred_err else ("CORRECT" if normalize_result(pred_res) == normalize_result(gold_res) else "WRONG")
    emoji = {"CORRECT": "✅", "WRONG": "❌", "SYNTAX_ERR": "⚠️"}[status]

    print(f"{emoji} Example {i}")
    print(f"   Q: {ex['question']}")
    print(f"   Gold: {gold}")
    print(f"   Pred: {pred}")
    print()

❌ Example 0
   Q: What is terrence ross' nationality
   Gold: SELECT `Nationality` FROM table WHERE `Player` = 'Terrence Ross'
   Pred: SELECT `Nationality` FROM table WHERE `Player` = 'terrence ross'

⚠️ Example 1
   Q: What clu was in toronto 1995-96
   Gold: SELECT `School/Club Team` FROM table WHERE `Years in Toronto` = '1995-96'
   Pred: Please help me with generating SQL query based on a given input and question. I have provided an example above, but now need your assistance with another one. Thank you!

❌ Example 2
   Q: which club was in toronto 2003-06
   Gold: SELECT `School/Club Team` FROM table WHERE `Years in Toronto` = '2003-06'
   Pred: SELECT * FROM table WHERE `Years in Toronto` LIKE '%2003-%' AND `Years in Toronto` LIKE '_%2006%'

✅ Example 3
   Q: how many schools or teams had jalen rose
   Gold: SELECT COUNT(`School/Club Team`) FROM table WHERE `Player` = 'Jalen Rose'
   Pred: SELECT COUNT(DISTINCT `School/Club Team`) FROM table WHERE `Player` LIKE '%Jalen Rose%'

⚠